In [2]:
from pyspark.sql import SparkSession
spark=(
    SparkSession
    .builder
    .appName("SparK first")
    .master("local[*]")
    .getOrCreate()
)
spark

In [6]:
# riders_df=spark.read.csv("E:\\Uber\\Uber\\Admin_data.csv",header=True, inferSchema=True)
rides_df=spark.read.csv("C:\\Users\\Administrator\\Downloads\\uber_rides.csv",header=True,inferSchema=True)
# customers_df=spark.read.csv("E:\\Uber\\Uber\\Customer_table.csv",header=True, inferSchema=True)

In [7]:

rides_df.printSchema()
# customers_df.printSchema()

root
 |-- Start_time: string (nullable = true)
 |-- End_time: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- Mobile: long (nullable = true)
 |-- Age: integer (nullable = true)
 |-- Pin-Codes: integer (nullable = true)
 |-- Source: string (nullable = true)
 |-- Vaccine_cus: string (nullable = true)
 |-- Destination: string (nullable = true)
 |-- Miles: double (nullable = true)
 |-- Est_Costing: double (nullable = true)
 |-- Ride_category: string (nullable = true)
 |-- Purpose: string (nullable = true)
 |-- temp: double (nullable = true)
 |-- clouds: double (nullable = true)
 |-- pressure: double (nullable = true)
 |-- humidity: double (nullable = true)
 |-- wind: double (nullable = true)
 |-- accquire_vehi: integer (nullable = true)
 |-- free_vehi: string (nullable = true)
 |-- Lattitute: double (nullable = true)
 |-- Longitude: double (nullable = true)
 |-- locationID: integer (nullable = true)
 |-- rating_cus: integer (nullable = true)
 |-- Driver_Name: str

# Q1. How many no of customers take trip from same location.

In [23]:
from pyspark.sql.functions import count,when,col,countDistinct

In [10]:
location_counts=rides_df.groupBy("Source").agg(count("*").alias("No_Of_Rides")).orderBy("No_Of_Rides",ascending=False)
location_counts.show(5)

+---------------+-----------+
|         Source|No_Of_Rides|
+---------------+-----------+
|    Fort Pierce|        108|
|        Midtown|         78|
|West Palm Beach|         54|
|           Cary|         52|
|Lower Manhattan|         26|
+---------------+-----------+
only showing top 5 rows



# Q2. what is priority for each ride category from each location.

In [11]:
priority_by_count=rides_df.groupBy("Source","Ride_category").agg(count("Ride_category").alias("No_of_trip"))\
.orderBy("Source","No_of_trip",ascending=[True, False])

priority_by_count.show(5)

+------+-------------+----------+
|Source|Ride_category|No_of_trip|
+------+-------------+----------+
|  Cary|         Auto|        12|
|  Cary|        Prime|        12|
|  Cary|    Uber-Mini|        12|
|  Cary|   Uber-Micro|         8|
|  Cary|         Bike|         8|
+------+-------------+----------+
only showing top 5 rows



# Q3.what are the longest locations of customer travelled.


In [13]:
longest_locations=rides_df.select("Source","Destination","Trip Distance")\
.where(col("Source")!=col("Destination"))\
.orderBy("Trip Distance",ascending=False)
longest_locations.show(5)

+------------+-----------+-------------+
|      Source|Destination|Trip Distance|
+------------+-----------+-------------+
|        Cary|Whitebridge|           80|
| East Harlem|Whitebridge|           80|
|    Elmhurst|       Cary|           80|
|Midtown East|     Durham|           80|
| Fort Pierce|       Cary|           80|
+------------+-----------+-------------+
only showing top 5 rows



# Q4.Drivers who completed ride with non -vaccinated customers


In [21]:
rides_df.filter(
    col("Vaccine_cus") == "NO"
).select("Driver_Name", "Vaccine_cus").distinct().show(5)

+-----------+-----------+
|Driver_Name|Vaccine_cus|
+-----------+-----------+
|       Dora|         NO|
|    Carolus|         NO|
|     Pattin|         NO|
|      Myrle|         NO|
|       Abbi|         NO|
+-----------+-----------+
only showing top 5 rows



# Q5.How many vaccinated customers have travelled.


In [25]:
unique_entries=rides_df.filter(col("Vaccine_cus")=="YES")
unique_entries.agg(
    countDistinct("customer_name").alias("Total_Customers")
).show()

+---------------+
|Total_Customers|
+---------------+
|            391|
+---------------+



# Q6.Customers who completed ride with non -vaccinated Drivers.

In [28]:
rides_df.filter(
    col("Vaccine_Driver")=="NO"
).select("customer_name","Vaccine_Driver").distinct().show(5)

+-------------+--------------+
|customer_name|Vaccine_Driver|
+-------------+--------------+
|       Davita|            NO|
|        Delly|            NO|
|          Gus|            NO|
|      Othello|            NO|
|      Abrahan|            NO|
+-------------+--------------+
only showing top 5 rows



# Q7.who is the customer completed highest no of rides

In [29]:
highest_traveled_customer=rides_df.groupBy("customer_name").agg(count("customer_name").alias("Trip Count"))\
.orderBy("Trip Count",ascending=False)

highest_traveled_customer.show(5)

+-------------+----------+
|customer_name|Trip Count|
+-------------+----------+
|      Mathian|         2|
|       Raeann|         2|
|       Teresa|         2|
|        Mayne|         2|
|    Westbrook|         2|
+-------------+----------+
only showing top 5 rows



# Q8. who is the driver completed highest no of rides.

In [32]:
rides_df.filter(
    col("Status")=="Completed"
).groupBy("Driver_Name").agg(
    count("Driver_Name").alias("Completed Rides")
).orderBy("Completed Rides",ascending=False).show(5)

+-----------+---------------+
|Driver_Name|Completed Rides|
+-----------+---------------+
|   Loutitia|              4|
|       Cher|              4|
|   Virginie|              4|
|     Shaine|              4|
|    Yasmeen|              4|
+-----------+---------------+
only showing top 5 rows



# Q9. what are first 10 age groups which uses uber services mostly

In [33]:
top_10_ages=rides_df.groupBy("Age").agg(count("*").alias("no_of_people")).orderBy("no_of_people",ascending=False)
top_10_ages.show(10)

+---+------------+
|Age|no_of_people|
+---+------------+
| 69|          16|
| 15|          15|
| 32|          15|
| 28|          12|
| 78|          11|
| 43|          11|
| 49|          11|
| 21|          11|
| 34|          10|
| 44|          10|
+---+------------+
only showing top 10 rows



# Q.10.what is the count of different destination locations from same start location and also
 # completed ride

In [35]:
diff_destinations=rides_df.select("Source","Destination","Status")\
.where((col("Source")!=col("Destination")) & (col("Status")=="Completed"))\
.orderBy(col("Source"))
diff_destinations.groupBy("Source").agg(count("*").alias("no_of_trips")).orderBy("no_of_trips",ascending=False).show()

+-----------------+-----------+
|           Source|no_of_trips|
+-----------------+-----------+
|      Fort Pierce|         47|
|          Midtown|         36|
|  West Palm Beach|         30|
|             Cary|         17|
|     Midtown East|         16|
|Flatiron District|         15|
|          Jamaica|         12|
|      East Harlem|         10|
|  Lower Manhattan|          9|
|    Hudson Square|          9|
|         New York|          8|
|         Elmhurst|          8|
+-----------------+-----------+



# Q11 what is the most expensive Drive.

In [37]:
most_expensive_drive=rides_df.select("Source","Destination","Final_cost").where(col("Source")!=col("Destination"))\
.orderBy(col("Final_cost").desc())\
.limit(1)
most_expensive_drive.show()

+-----------+-----------+----------+
|     Source|Destination|Final_cost|
+-----------+-----------+----------+
|Fort Pierce|       Katy|    3060.0|
+-----------+-----------+----------+



In [20]:
riders_df.select("Vaccine_cus","Vaccine_Ri").show()

+-----------+----------+
|Vaccine_cus|Vaccine_Ri|
+-----------+----------+
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|         NO|        NO|
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|         NO|        NO|
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|         NO|        NO|
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|        YES|       YES|
|         NO|        NO|
+-----------+----------+
only showing top 20 rows

